In [ ]:
import os
import torch
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt


assert torch.cuda.is_available(), "GPU is required to run this notebook."

OUTPUT_DIR = "../../results/lora_seed_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Environment ready")
print("📁 Output directory:", OUTPUT_DIR)


In [ ]:
# =========================
# LOAD MODEL + LoRA
# =========================
BASE_MODEL = "runwayml/stable-diffusion-v1-5"
LORA_DIR = ".../megamendung"
LORA_FILE = "(...).safetensors" # Ex. pytorch_lora_weights.safetensors

pipe_lora = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16
).to("cuda")

pipe_lora.load_lora_weights(
    LORA_DIR,
    weight_name=LORA_FILE
)

pipe_lora.enable_attention_slicing()

print("✅ LoRA loaded correctly (local path)")


In [ ]:
# =========================
# CONFIG
# =========================
prompt = "traditional batik megamendung textile pattern"

seeds = [42, 123, 999]
images = []

# =========================
# GENERATE IMAGES
# =========================
for seed in seeds:
    generator = torch.Generator("cuda").manual_seed(seed)

    result = pipe_lora(
        prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator
    )
    result.images[0].save(f"{OUTPUT_DIR}/diversity_seed_{seed}.png")
    images.append(result.images[0])


# =========================
# PLOT RESULTS
# =========================
plt.figure(figsize=(15, 5))

for i, img in enumerate(images):
    plt.subplot(1, 3, i + 1)
    plt.imshow(img)
    plt.title(f"Seed = {seeds[i]}")
    plt.axis("off")

plt.suptitle("Same Prompt, Different Seeds (LoRA Enabled)", fontsize=14)
plt.tight_layout()
plt.show()